# 컨텍스트 윈도

모델에 넣을 수 있는 토큰 예산은 입력과 출력 사이에서 어떻게 나눌까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 입력 토큰 세기

Qwen2.5 로컬 토큰화기로 시스템 지시, 문서, 질문을 함께 세어 봅니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision="7ae557604adf67be50417f59c2c2f167def9a775")
parts = {
    "시스템": "주어진 문서 범위에서만 답하세요.",
    "문서": "환불은 결제일로부터 7일 이내에 신청합니다. " * 20,
    "질문": "환불은 며칠 이내에 신청하나요?",
}
counts = {name: len(tokenizer.encode(text)) for name, text in parts.items()}
print(counts)
print("raw 텍스트 합계:", sum(counts.values()))
messages = [
    {"role": "system", "content": parts["시스템"]},
    {"role": "user", "content": parts["문서"] + "\n\n" + parts["질문"]},
]
actual_input = len(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True))
print("채팅 템플릿 적용 입력:", actual_input)

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part1/ch05-llm-principles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'시스템': 12, '문서': 360, '질문': 15}
raw 텍스트 합계: 387
채팅 템플릿 적용 입력: 400


## 출력 공간 예약

실습용 작은 한도를 정하고 출력에 남겨 둘 토큰을 계산합니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
window = 512
for output_budget in [32, 64, 128]:
    input_budget = window - output_budget
    remaining = input_budget - actual_input
    print(f"출력 {output_budget:3}개 예약 -> 입력 한도 {input_budget:3}개, 남은 입력 {remaining:3}개")

출력  32개 예약 -> 입력 한도 480개, 남은 입력  80개
출력  64개 예약 -> 입력 한도 448개, 남은 입력  48개
출력 128개 예약 -> 입력 한도 384개, 남은 입력 -16개


## 문서 길이 비교

동일한 문장을 늘려 언제 실습용 한도를 넘는지 관찰합니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
full_counts = {}
for repeat in [10, 20, 40, 80]:
    document = "환불은 결제일로부터 7일 이내에 신청합니다. " * repeat
    messages = [
        {"role": "system", "content": parts["시스템"]},
        {"role": "user", "content": document + "\n\n" + parts["질문"]},
    ]
    used = len(tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True))
    full_counts[repeat] = used
    print(repeat, "반복:", used, "토큰 | 448 이하:", used <= 448)

10 반복: 220 토큰 | 448 이하: True
20 반복: 400 토큰 | 448 이하: True
40 반복: 760 토큰 | 448 이하: False
80 반복: 1480 토큰 | 448 이하: False


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [4]:
assert sum(counts.values()) == 387 and actual_input == 400
assert full_counts[20] <= 448 < full_counts[40]
print("검증: 채팅 템플릿 입력과 20/40회 반복의 한도 분류를 확인했습니다.")

검증: 채팅 템플릿 입력과 20/40회 반복의 한도 분류를 확인했습니다.
